# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminamandzukic/FlyRank_starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will treat this as a binary classification and ranking problem. The target indicates whether a content item experiences a meaningful decline in search visibility during the following outcome window.

I will first train a Logistic Regression model as a simple and interpretable model, followed by a Random Forest to test whether nonlinear relationships between historical search signals improve prediction.

The final output will use predicted decline probabilities as ranking scores rather than only predicted class labels. This fits the project because the goal is to prioritize which content items should be reviewed first.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use a time-aware split because the model is intended to predict future decline from past search-performance signals.

Training examples will come from earlier periods, validation examples will come from a later development period, and the final June 2026 outcome period will be kept sealed until the model and feature choices are finalized.

This avoids randomly mixing earlier and later observations, which could make the evaluation unrealistically easy.

The same time split will be used for both the learned models and the Week-4 rule-based baseline so that their results are directly comparable.

The exact monthly split is:

- Training: previous-month features predicting outcomes from March 2025 through April 2026.
- Validation: April 2026 features predicting May 2026 outcomes.
- Final sealed test: May 2026 features predicting June 2026 outcomes.

January 2025 is not used as a feature month because the warehouse begins partway through that month. Average daily impressions will be used when comparing months so that months with different numbers of days remain comparable.

In [44]:
from huggingface_hub import list_repo_files
import getpass

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ").strip()

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

months = sorted({
    path.split("/")[1]
    for path in files
    if path.startswith("fact_content_daily_performance/month=")
})

print("Available monthly partitions:")
for month in months:
    print(month)

Available monthly partitions:
month=2025-01
month=2025-02
month=2025-03
month=2025-04
month=2025-05
month=2025-06
month=2025-07
month=2025-08
month=2025-09
month=2025-10
month=2025-11
month=2025-12
month=2026-01
month=2026-02
month=2026-03
month=2026-04
month=2026-05
month=2026-06


In [45]:
import duckdb
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=HF_TOKEN
)

con = duckdb.connect()

window_check = con.sql(f"""
    SELECT
        'March 2026' AS month,
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{march_path}')

    UNION ALL

    SELECT
        'April 2026' AS month,
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{april_path}')
""").df()

window_check

,month,rows,min_date,max_date
0,March 2026,9841378,2026-03-01,2026-03-31
1,April 2026,10424730,2026-04-01,2026-04-30


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [46]:
model_frame = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_feature,
            SUM(gsc_clicks) AS clicks_feature,
            AVG(gsc_avg_position) AS avg_position_feature

        FROM read_parquet('{march_path}')
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_outcome

        FROM read_parquet('{april_path}')
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,

        m.impressions_feature,
        m.clicks_feature,
        m.avg_position_feature,

        a.impressions_outcome

    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
""").df()

model_frame["avg_daily_impressions_feature"] = (
    model_frame["impressions_feature"] / 31
)

model_frame["avg_daily_impressions_outcome"] = (
    model_frame["impressions_outcome"] / 30
)

model_frame["is_declining"] = (
    model_frame["avg_daily_impressions_outcome"]
    <
    0.8 * model_frame["avg_daily_impressions_feature"]
).astype(int)

print("Rows:", len(model_frame))
print("Declining share:", round(model_frame["is_declining"].mean(), 3))

model_frame.head()

Rows: 331436
Declining share: 0.276


,client_hash_id,content_hash_id,impressions_feature,clicks_feature,avg_position_feature,impressions_outcome,avg_daily_impressions_feature,avg_daily_impressions_outcome,is_declining
0,client_62f4a7e64f5e0096,content_19c7a0763227afca,102.0,0.0,6.326374,18.0,3.290323,0.600000,1
1,client_62f4a7e64f5e0096,content_d61de87e2d5afcbc,13287.0,8.0,4.955010,12489.0,428.612903,416.300000,0
2,client_62f4a7e64f5e0096,content_cfa688665f1b4dad,23824.0,9.0,3.588213,20568.0,768.516129,685.600000,0
3,client_62f4a7e64f5e0096,content_b09b485e08454deb,171.0,0.0,23.194897,107.0,5.516129,3.566667,1
4,client_62f4a7e64f5e0096,content_94a471496f1886b7,146.0,0.0,15.916334,37.0,4.709677,1.233333,1


In [47]:
eligible_frame = model_frame[
    model_frame["impressions_feature"] >= 200
].copy()

print("Rows before threshold:", len(model_frame))
print("Rows after threshold:", len(eligible_frame))
print(
    "Declining share after threshold:",
    round(eligible_frame["is_declining"].mean(), 3)
)

Rows before threshold: 331436
Rows after threshold: 84833
Declining share after threshold: 0.491


In [48]:
from huggingface_hub import hf_hub_download

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN
)

client_history = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        MIN(gsc_data_start) AS earliest_start,
        MAX(gsc_data_start) AS latest_start,
        COUNT(*) FILTER (
            WHERE gsc_data_start <= DATE '2026-03-01'
        ) AS clients_with_full_march_history
    FROM read_parquet('{clients_path}')
""").df()

client_history

,total_clients,earliest_start,latest_start,clients_with_full_march_history
0,104,2025-01-27,2026-06-02,52


In [49]:
eligible_full_history = con.sql(f"""
    SELECT e.*
    FROM eligible_frame e
    INNER JOIN read_parquet('{clients_path}') c
        ON e.client_hash_id = c.client_hash_id
    WHERE c.gsc_data_start <= DATE '2026-03-01'
""").df()

print("Rows before client-history filter:", len(eligible_frame))
print("Rows after client-history filter:", len(eligible_full_history))
print(
    "Declining share:",
    round(eligible_full_history["is_declining"].mean(), 3)
)

print(
    "Clients remaining:",
    eligible_full_history["client_hash_id"].nunique()
)

Rows before client-history filter: 84833
Rows after client-history filter: 84656
Declining share: 0.492
Clients remaining: 37


In [50]:
eligible_full_history["ctr_feature"] = (
    eligible_full_history["clicks_feature"]
    / eligible_full_history["impressions_feature"]
)

feature_cols = [
    "impressions_feature",
    "clicks_feature",
    "avg_position_feature",
    "ctr_feature",
]

print("Features:")
for col in feature_cols:
    print("-", col)

eligible_full_history[feature_cols + ["is_declining"]].head()

Features:
- impressions_feature
- clicks_feature
- avg_position_feature
- ctr_feature


,impressions_feature,clicks_feature,avg_position_feature,ctr_feature,is_declining
0,13287.0,8.0,4.955010,0.000602,0
1,23824.0,9.0,3.588213,0.000378,0
2,1532.0,0.0,4.110906,0.000000,0
3,209.0,0.0,3.232526,0.000000,0
4,1424.0,2.0,3.144647,0.001404,1


For the first learned models, I use four historical GSC features: impressions, clicks, average search position, and CTR. I exclude pseudonymous IDs and all outcome-window fields from the feature set.

I leave GA4 features out of the primary model because analytics availability is sparse in the warehouse and treating unavailable analytics as zero would be misleading.

In [51]:
import calendar
import pandas as pd
from huggingface_hub import hf_hub_download


def build_month_pair(feature_month, outcome_month):
    feature_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={feature_month}/data_0.parquet",
        token=HF_TOKEN
    )

    outcome_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={outcome_month}/data_0.parquet",
        token=HF_TOKEN
    )

    feature_year, feature_num = map(int, feature_month.split("-"))
    outcome_year, outcome_num = map(int, outcome_month.split("-"))

    feature_days = calendar.monthrange(feature_year, feature_num)[1]
    outcome_days = calendar.monthrange(outcome_year, outcome_num)[1]

    feature_start = f"{feature_month}-01"

    frame = con.sql(f"""
        WITH features AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_feature,
                SUM(gsc_clicks) AS clicks_feature,
                AVG(gsc_avg_position) AS avg_position_feature
            FROM read_parquet('{feature_path}')
            GROUP BY
                client_hash_id,
                content_hash_id
        ),

        outcomes AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_outcome
            FROM read_parquet('{outcome_path}')
            GROUP BY
                client_hash_id,
                content_hash_id
        )

        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.impressions_feature,
            f.clicks_feature,
            f.avg_position_feature,
            o.impressions_outcome
        FROM features f
        INNER JOIN outcomes o
            ON f.client_hash_id = o.client_hash_id
           AND f.content_hash_id = o.content_hash_id
        INNER JOIN read_parquet('{clients_path}') c
            ON f.client_hash_id = c.client_hash_id
        WHERE
            f.impressions_feature >= 200
            AND c.gsc_data_start <= DATE '{feature_start}'
    """).df()

    frame["ctr_feature"] = (
        frame["clicks_feature"] /
        frame["impressions_feature"]
    )

    frame["avg_daily_impressions_feature"] = (
        frame["impressions_feature"] / feature_days
    )

    frame["avg_daily_impressions_outcome"] = (
        frame["impressions_outcome"] / outcome_days
    )

    frame["is_declining"] = (
        frame["avg_daily_impressions_outcome"]
        < 0.8 * frame["avg_daily_impressions_feature"]
    ).astype(int)

    frame["feature_month"] = feature_month
    frame["outcome_month"] = outcome_month

    return frame

In [52]:
march_april = build_month_pair("2026-03", "2026-04")

print("Rows:", len(march_april))
print("Clients:", march_april["client_hash_id"].nunique())
print("Declining share:", round(march_april["is_declining"].mean(), 3))

Rows: 84656
Clients: 37
Declining share: 0.492


In [53]:
april_may = build_month_pair("2026-04", "2026-05")

print("Validation rows:", len(april_may))
print("Validation clients:", april_may["client_hash_id"].nunique())
print(
    "Validation declining share:",
    round(april_may["is_declining"].mean(), 3)
)

Validation rows: 86282
Validation clients: 42
Validation declining share: 0.576


In [54]:
training_pairs = [
    ("2025-02", "2025-03"),
    ("2025-03", "2025-04"),
    ("2025-04", "2025-05"),
    ("2025-05", "2025-06"),
    ("2025-06", "2025-07"),
    ("2025-07", "2025-08"),
    ("2025-08", "2025-09"),
    ("2025-09", "2025-10"),
    ("2025-10", "2025-11"),
    ("2025-11", "2025-12"),
    ("2025-12", "2026-01"),
    ("2026-01", "2026-02"),
    ("2026-02", "2026-03"),
    ("2026-03", "2026-04"),
]

training_frames = []

for feature_month, outcome_month in training_pairs:
    frame = build_month_pair(feature_month, outcome_month)

    print(
        f"{feature_month} -> {outcome_month}:",
        len(frame),
        "rows"
    )

    training_frames.append(frame)

train_frame = pd.concat(
    training_frames,
    ignore_index=True
)

print()
print("Total training rows:", len(train_frame))
print("Training clients:", train_frame["client_hash_id"].nunique())
print(
    "Training declining share:",
    round(train_frame["is_declining"].mean(), 3)
)

2025-02 -> 2025-03: 782 rows
2025-03 -> 2025-04: 3436 rows
2025-04 -> 2025-05: 6194 rows
2025-05 -> 2025-06: 7797 rows
2025-06 -> 2025-07: 7475 rows
2025-07 -> 2025-08: 10646 rows
2025-08 -> 2025-09: 15161 rows
2025-09 -> 2025-10: 16072 rows
2025-10 -> 2025-11: 22992 rows
2025-11 -> 2025-12: 29322 rows
2025-12 -> 2026-01: 43621 rows
2026-01 -> 2026-02: 51975 rows
2026-02 -> 2026-03: 58369 rows
2026-03 -> 2026-04: 84656 rows

Total training rows: 358498
Training clients: 42
Training declining share: 0.303


In [55]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feature_cols = [
    "impressions_feature",
    "clicks_feature",
    "avg_position_feature",
    "ctr_feature",
]

X_train = train_frame[feature_cols]
y_train = train_frame["is_declining"]

X_val = april_may[feature_cols]
y_val = april_may["is_declining"]

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

val_prob_logistic = logistic_model.predict_proba(X_val)[:, 1]

logistic_auc = roc_auc_score(
    y_val,
    val_prob_logistic
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Validation base rate:", round(y_val.mean(), 3))
print("Logistic Regression ROC-AUC:", round(logistic_auc, 3))

Training rows: 358498
Validation rows: 86282
Validation base rate: 0.576
Logistic Regression ROC-AUC: 0.588


In [56]:
import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1]
    top_k = y_true.iloc[order[:k]]
    return top_k.mean()

for k in [20, 50, 100, 500]:
    p_at_k = precision_at_k(
        y_val.reset_index(drop=True),
        val_prob_logistic,
        k
    )

    print(
        f"Precision@{k}:",
        round(p_at_k, 3)
    )

print("Validation base rate:", round(y_val.mean(), 3))

Precision@20: 0.55
Precision@50: 0.54
Precision@100: 0.55
Precision@500: 0.504
Validation base rate: 0.576


In [57]:
import numpy as np

validation_compare = april_may.copy()

# Week-4 baseline (at least 200 impressions is already enforced by build_month_pair(); pages with average position > 10 qualify)
validation_compare["baseline_score"] = np.where(
    validation_compare["avg_position_feature"] > 10,
    validation_compare["impressions_feature"],
    0
)

baseline_scores = validation_compare["baseline_score"].to_numpy()

print("Week-4 baseline")

for k in [20, 50, 100, 500]:
    p_at_k = precision_at_k(
        validation_compare["is_declining"].reset_index(drop=True),
        baseline_scores,
        k
    )

    print(f"Precision@{k}:", round(p_at_k, 3))

print(
    "Validation base rate:",
    round(validation_compare["is_declining"].mean(), 3)
)

Week-4 baseline
Precision@20: 0.6
Precision@50: 0.66
Precision@100: 0.7
Precision@500: 0.666
Validation base rate: 0.576


In [58]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

rf_model.fit(X_train, y_train)

val_prob_rf = rf_model.predict_proba(X_val)[:, 1]

rf_auc = roc_auc_score(
    y_val,
    val_prob_rf
)

print("Random Forest ROC-AUC:", round(rf_auc, 3))

for k in [20, 50, 100, 500]:
    p_at_k = precision_at_k(
        y_val.reset_index(drop=True),
        val_prob_rf,
        k
    )

    print(f"Precision@{k}:", round(p_at_k, 3))

print("Validation base rate:", round(y_val.mean(), 3))

Random Forest ROC-AUC: 0.538
Precision@20: 0.7
Precision@50: 0.72
Precision@100: 0.59
Precision@500: 0.594
Validation base rate: 0.576


In [59]:
def precision_values(y_true, scores):
    return {
        20: precision_at_k(y_true.reset_index(drop=True), scores, 20),
        50: precision_at_k(y_true.reset_index(drop=True), scores, 50),
        100: precision_at_k(y_true.reset_index(drop=True), scores, 100),
        500: precision_at_k(y_true.reset_index(drop=True), scores, 500),
    }

baseline_precision = precision_values(
    y_val,
    baseline_scores
)

logistic_precision = precision_values(
    y_val,
    val_prob_logistic
)

rf_precision = precision_values(
    y_val,
    val_prob_rf
)

baseline_auc = roc_auc_score(
    y_val,
    baseline_scores
)

comparison_table = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "ROC-AUC": [
        baseline_auc,
        logistic_auc,
        rf_auc
    ],
    "Precision@20": [
        baseline_precision[20],
        logistic_precision[20],
        rf_precision[20]
    ],
    "Precision@50": [
        baseline_precision[50],
        logistic_precision[50],
        rf_precision[50]
    ],
    "Precision@100": [
        baseline_precision[100],
        logistic_precision[100],
        rf_precision[100]
    ],
    "Precision@500": [
        baseline_precision[500],
        logistic_precision[500],
        rf_precision[500]
    ]
})

comparison_table.round(3)

,Method,ROC-AUC,Precision@20,Precision@50,Precision@100,Precision@500
0,Week-4 baseline,0.554,0.60,0.66,0.70,0.666
1,Logistic Regression,0.588,0.55,0.54,0.55,0.504
2,Random Forest,0.538,0.70,0.72,0.59,0.594


On the April-May validation period, the Week-4 rule-based baseline outperformed both learned models at the top of the ranked queue.

Logistic Regression achieved a ROC-AUC of 0.588, while Random Forest achieved 0.538. However, both models had weaker Precision@K than the transparent baseline for the highest-priority review positions.

This suggests that the current four-feature model does not improve on the simpler rule for this validation period. I therefore keep the rule-based baseline as an important reference rather than claiming that the learned model is better.

## 4. Errors and interpretation

The learned models did not consistently outperform the transparent Week-4 baseline on the validation period.

Logistic Regression showed some overall discrimination, but its highest-ranked pages had lower precision than the baseline. Random Forest performed slightly better than Logistic Regression at larger K values, but it still did not beat the baseline for the highest-priority review positions.

This suggests that the current historical features contain some predictive signal, but the simple visibility-and-position rule remains more effective for prioritizing the very top of the queue.

In [60]:
import pandas as pd

# Logistic Regression coefficients
logistic_coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logistic_model.named_steps["model"].coef_[0]
}).sort_values(
    "coefficient",
    key=abs,
    ascending=False
)

# Random Forest feature importances
rf_importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Logistic Regression coefficients")
display(logistic_coefficients)

print("Random Forest feature importances")
display(rf_importances)

Logistic Regression coefficients


,feature,coefficient
3,ctr_feature,-0.335700
2,avg_position_feature,0.247786
0,impressions_feature,0.127360
1,clicks_feature,-0.103851


Random Forest feature importances


,feature,importance
2,avg_position_feature,0.503451
0,impressions_feature,0.271098
3,ctr_feature,0.186374
1,clicks_feature,0.039077


In [61]:
val_errors = april_may.copy()

val_errors["rf_prob"] = val_prob_rf
val_errors["rf_pred"] = (val_errors["rf_prob"] >= 0.5).astype(int)

false_positives = val_errors[
    (val_errors["rf_pred"] == 1) &
    (val_errors["is_declining"] == 0)
].sort_values("rf_prob", ascending=False)

false_negatives = val_errors[
    (val_errors["rf_pred"] == 0) &
    (val_errors["is_declining"] == 1)
].sort_values("rf_prob")

print("False positives:")
display(
    false_positives[
        feature_cols + ["rf_prob", "is_declining"]
    ].head(3)
)

print("False negatives:")
display(
    false_negatives[
        feature_cols + ["rf_prob", "is_declining"]
    ].head(3)
)

False positives:


,impressions_feature,clicks_feature,avg_position_feature,ctr_feature,rf_prob,is_declining
73733,350.0,0.0,59.734290,0.0,0.99,0
34801,1242.0,0.0,66.973610,0.0,0.98,0
25207,494.0,0.0,10.781333,0.0,0.98,0


False negatives:


,impressions_feature,clicks_feature,avg_position_feature,ctr_feature,rf_prob,is_declining
72400,338.0,1.0,4.658657,0.002959,0.0,1
41275,238.0,2.0,26.449463,0.008403,0.0,1
76443,734.0,1.0,21.638374,0.001362,0.0,1


The models appear to rely most strongly on historical search position, impressions, and CTR.

For Logistic Regression, lower CTR is associated with higher predicted decline risk, while worse average search position is associated with higher risk. Random Forest also relies most heavily on average search position, followed by impressions and CTR.

The error cases show that similar historical search profiles can lead to different future outcomes. This helps explain why the learned models struggle to consistently outperform the simpler Week-4 rule. These features contain useful signal, but they do not fully capture the factors that determine future visibility changes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.